In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    for path in [current, *current.parents]:
        if (path / "Data").exists():
            return path
    raise FileNotFoundError("Could not find project root containing Data/")


PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "Data"
DATA_FILE = DATA_DIR / "final_trip_data.parquet"
# DATA_FILE = DATA_DIR / "artefakty_100000_min_full.csv"
rides_data = pd.read_parquet(DATA_FILE)

In [3]:
rides_data.head()

,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_ms,start_lat,start_lon,end_lat,end_lon
0,145207079,2024-12-14 23:59:00,300083,"Duke Street Hill, London Bridge",2024-12-14 23:59:00,300083.0,"Duke Street Hill, London Bridge",57763.0,CLASSIC,29s,29203.0,51.50630441,-0.087262995,51.50630441,-0.087262995
1,145207080,2024-12-14 23:59:00,1133,"Baylis Road, Waterloo",2024-12-15 00:26:00,200011.0,"Furze Green, Bow",62625.0,PBSC_EBIKE,27m 0s,1620164.0,51.50144456,-0.110699309,51.519265,-0.021345
2,145207081,2024-12-14 23:59:00,3447,"Gloucester Road (North), Kensington",2024-12-15 00:17:00,200181.0,"Richmond Way, Shepherd's Bush",62291.0,PBSC_EBIKE,18m 16s,1096981.0,51.49792478,-0.183834706,51.50035306,-0.217515071
3,145207082,2024-12-14 23:59:00,1112,"Nutford Place, Marylebone",2024-12-15 00:12:00,3422.0,"Charlbert Street, St. John's Wood",50575.0,CLASSIC,13m 3s,783763.0,51.5165179,-0.164393768,51.53430039,-0.1680743
4,145207083,2024-12-14 23:59:00,1122,"Ashley Place, Victoria",2024-12-15 00:04:00,200048.0,"Page Street, Westminster",52143.0,CLASSIC,5m 22s,322155.0,51.49616092,-0.140947636,51.493978,-0.127554


In [4]:
bike_counts = rides_data["bike_id"].value_counts()

bike_counts

bike_id
58726.0    2704
59402.0    2623
59896.0    2600
59889.0    2537
59880.0    2514
           ... 
99975.0       1
99997.0       1
60486.0       1
21823.0       1
99987.0       1
Name: count, Length: 15739, dtype: int64

In [5]:
selected_bike_id = 58726.0
one_bike_data = rides_data.loc[
    rides_data["bike_id"] == selected_bike_id
].copy()

one_bike_data

,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_ms,start_lat,start_lon,end_lat,end_lon
25077,145181354,2024-12-13 16:56:00,1216,"Swan Street, The Borough",2024-12-13 17:04:00,1219.0,"Lower Marsh, Waterloo",58726.0,CLASSIC,8m 0s,480088.0,51.50029631,-0.092762704,51.5001394,-0.11393599999999537
27133,145179292,2024-12-13 15:12:00,974,"Guilford Street , Bloomsbury",2024-12-13 15:31:00,1216.0,"Swan Street, The Borough",58726.0,CLASSIC,19m 3s,1143019.0,51.52334672,-0.120202614,51.50029631,-0.092762704
27317,145179093,2024-12-13 15:01:00,1010,"Cartwright Gardens , Bloomsbury",2024-12-13 15:06:00,974.0,"Guilford Street , Bloomsbury",58726.0,CLASSIC,4m 30s,270808.0,51.52635795,-0.125979294,51.52334672,-0.120202614
28087,145178314,2024-12-13 14:12:00,2665,"Lord's, St. John's Wood",2024-12-13 14:32:00,1010.0,"Cartwright Gardens , Bloomsbury",58726.0,CLASSIC,20m 4s,1204076.0,51.5291212008901,-0.171185284853,51.52635795,-0.125979294
30607,145175732,2024-12-13 11:48:00,3422,"Charlbert Street, St. John's Wood",2024-12-13 11:52:00,2665.0,"Lord's, St. John's Wood",58726.0,CLASSIC,3m 35s,215501.0,51.53430039,-0.1680743,51.5291212008901,-0.171185284853
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17798185,146344838,2025-02-16 17:17:00,300203,"Euston Square Gardens, Euston",2025-02-16 18:04:00,200229.0,"Fulham Park Road, Fulham",58726.0,CLASSIC,47m 42s,2862559.0,51.527068,-0.13186102,51.473471,-0.20782
17801135,146341874,2025-02-16 14:46:00,1049,"Finsbury Leisure Centre, St. Luke's",2025-02-16 14:59:00,300203.0,"Euston Square Gardens, Euston",58726.0,CLASSIC,12m 53s,773662.0,51.52600832,-0.096317627,51.527068,-0.13186102
17816102,146326436,2025-02-15 13:42:00,3469,"Cadogan Place, Knightsbridge",2025-02-15 14:09:00,960.0,"Hop Exchange, The Borough",58726.0,CLASSIC,26m 20s,1580528.0,51.49464523,-0.158105512,51.50462759,-0.091773776
17817214,146325294,2025-02-15 12:51:00,300086,"Fulham Broadway, Walham Green",2025-02-15 13:07:00,3469.0,"Cadogan Place, Knightsbridge",58726.0,CLASSIC,15m 17s,917096.0,51.47993289,-0.19411695,51.49464523,-0.158105512


In [6]:
# ---------- Straight-line distance between start and end stations ----------

distance_columns = [
    "trip_id",
    "start_date",
    "end_date",
    "total_duration_ms",
    "start_lat",
    "start_lon",
    "end_lat",
    "end_lon"
]

ride_distance_data = rides_data[distance_columns].copy()
ride_distance_data["total_duration_minutes"] = ride_distance_data["total_duration_ms"] / 60000

coordinate_columns = ["start_lat", "start_lon", "end_lat", "end_lon"]

for coordinate_column in coordinate_columns:
    ride_distance_data[coordinate_column] = pd.to_numeric(
        ride_distance_data[coordinate_column],
        errors="coerce"
    )

earth_radius_km = 6371.0088

start_lat_rad = np.radians(ride_distance_data["start_lat"])
start_lon_rad = np.radians(ride_distance_data["start_lon"])
end_lat_rad = np.radians(ride_distance_data["end_lat"])
end_lon_rad = np.radians(ride_distance_data["end_lon"])

delta_lat = end_lat_rad - start_lat_rad
delta_lon = end_lon_rad - start_lon_rad

haversine_a = (
    np.sin(delta_lat / 2) ** 2
    + np.cos(start_lat_rad) * np.cos(end_lat_rad) * np.sin(delta_lon / 2) ** 2
)

haversine_a = np.clip(haversine_a, 0, 1)

ride_distance_data["straight_line_distance_km"] = (
    2 * earth_radius_km * np.arcsin(np.sqrt(haversine_a))
)

missing_coordinates_count = ride_distance_data[coordinate_columns].isna().any(axis=1).sum()

print(f"Liczba przejazdów: {len(ride_distance_data)}")
print(f"Liczba przejazdów z brakującymi współrzędnymi: {missing_coordinates_count}")

display(
    ride_distance_data["straight_line_distance_km"].describe(
        percentiles=[0.05, 0.5, 0.9, 0.95, 0.99]
    )
)

display(
    ride_distance_data[
        [
            "trip_id",
            "start_date",
            "end_date",
            "total_duration_minutes",
            "start_lat",
            "start_lon",
            "end_lat",
            "end_lon",
            "straight_line_distance_km"
        ]
    ]
    .sort_values("straight_line_distance_km", ascending=False)
    .head()
)

Liczba przejazdów: 17426293
Liczba przejazdów z brakującymi współrzędnymi: 0


count    1.742629e+07
mean     2.378230e+00
std      1.736340e+00
min      0.000000e+00
5%       3.420790e-01
50%      1.985467e+00
90%      4.702962e+00
95%      5.774866e+00
99%      8.081387e+00
max      1.696891e+01
Name: straight_line_distance_km, dtype: float64

,trip_id,start_date,end_date,total_duration_minutes,start_lat,start_lon,end_lat,end_lon,straight_line_distance_km
3583685,137489640,2024-02-24 12:20:00,2024-02-24 14:06:00,105.405000,51.546326,-0.009935,51.463211,-0.215551,16.968915
3583672,137489653,2024-02-24 12:20:00,2024-02-24 14:06:00,105.153683,51.546326,-0.009935,51.463211,-0.215551,16.968915
702195,140275736,2024-06-17 18:35:00,2024-06-17 19:40:00,65.797917,51.464786,-0.215619,51.541793,-0.004810,16.918216
8381584,140191949,2024-06-14 15:26:00,2024-06-14 16:26:00,60.530150,51.494224,-0.236770,51.541793,-0.004810,16.899189
14062480,151049463,2025-08-07 17:21:00,2025-08-07 18:50:00,88.415300,51.494224,-0.236770,51.541793,-0.004810,16.899189


In [7]:
one_bike_data = (
    rides_data.loc[rides_data["bike_id"] == selected_bike_id]
    .merge(
        ride_distance_data[["trip_id", "total_duration_minutes", "straight_line_distance_km"]],
        on="trip_id",
        how="left"
    )
    .copy()
)

In [8]:
duration_hours = one_bike_data["total_duration_minutes"] / 60

one_bike_data["speed_km_per_h"] = np.where(
    duration_hours > 0,
    one_bike_data["straight_line_distance_km"] / duration_hours,
    np.nan
)

In [9]:
one_bike_data = one_bike_data.sort_values(
    ["start_date", "end_date", "trip_id"]
).reset_index(drop=True)

In [10]:
one_bike_data.head()

,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_ms,start_lat,start_lon,end_lat,end_lon,total_duration_minutes,straight_line_distance_km,speed_km_per_h
0,136454275,2024-01-01 11:00:00,1143,"Kensington Church Street, Kensington",2024-01-01 11:11:00,2585.0,"Orsett Terrace, Bayswater",58726.0,CLASSIC,11m 0s,660695.0,51.50315739,-0.191496313,51.517932,-0.183716959,11.011583,1.728826,9.420042
1,136471726,2024-01-03 08:28:00,2585,"Orsett Terrace, Bayswater",2024-01-03 08:42:00,3504.0,"Moor Street, Soho",58726.0,CLASSIC,13m 51s,831512.0,51.517932,-0.183716959,51.51352755,-0.130110822,13.858533,3.741558,16.198935
2,136473191,2024-01-03 09:12:00,3504,"Moor Street, Soho",2024-01-03 09:32:00,1055.0,"Wellington Road, St. John's Wood",58726.0,CLASSIC,20m 33s,1233325.0,51.51352755,-0.130110822,51.53304322,-0.172528678,20.555417,3.649862,10.653723
3,136474464,2024-01-03 10:40:00,1055,"Wellington Road, St. John's Wood",2024-01-03 10:59:00,3504.0,"Moor Street, Soho",58726.0,CLASSIC,18m 36s,1116471.0,51.53304322,-0.172528678,51.51352755,-0.130110822,18.607850,3.649862,11.768781
4,136477566,2024-01-03 15:21:00,3504,"Moor Street, Soho",2024-01-03 15:28:00,1159.0,"Berry Street, Clerkenwell",58726.0,CLASSIC,7m 4s,424204.0,51.51352755,-0.130110822,51.52285301,-0.099994052,7.070067,2.327608,19.753207


In [11]:
# ---------- Station continuity between consecutive trips of one bike ----------

bike_id_to_check = selected_bike_id

def find_station_continuity_breaks(bike_id, rides=rides_data):
    bike_trips = (
        rides.loc[rides["bike_id"] == bike_id]
        .sort_values(["start_date", "end_date", "trip_id"])
        .copy()
    )

    for coordinate_column in ["start_lat", "start_lon", "end_lat", "end_lon"]:
        bike_trips[coordinate_column] = pd.to_numeric(
            bike_trips[coordinate_column],
            errors="coerce"
        )

    bike_trips["previous_trip_id"] = bike_trips["trip_id"].shift()
    bike_trips["previous_end_date"] = bike_trips["end_date"].shift()
    bike_trips["previous_end_station_id"] = bike_trips["end_station_id"].shift()
    bike_trips["previous_end_station_name"] = bike_trips["end_station_name"].shift()
    bike_trips["previous_end_lat"] = bike_trips["end_lat"].shift()
    bike_trips["previous_end_lon"] = bike_trips["end_lon"].shift()
    bike_trips["time_since_previous_trip"] = (
        bike_trips["start_date"] - bike_trips["previous_end_date"]
    )

    current_start_station_id = pd.to_numeric(
        bike_trips["start_station_id"],
        errors="coerce"
    )
    previous_end_station_id = pd.to_numeric(
        bike_trips["previous_end_station_id"],
        errors="coerce"
    )

    mismatch_mask = (
        bike_trips["previous_trip_id"].notna()
        & current_start_station_id.notna()
        & previous_end_station_id.notna()
        & (current_start_station_id != previous_end_station_id)
    )

    continuity_columns = [
        "previous_trip_id",
        "previous_end_date",
        "previous_end_station_id",
        "previous_end_station_name",
        "previous_end_lat",
        "previous_end_lon",
        "trip_id",
        "start_date",
        "start_station_id",
        "start_station_name",
        "start_lat",
        "start_lon",
        "end_date",
        "end_station_id",
        "end_station_name",
        "time_since_previous_trip",
    ]

    return bike_trips.loc[mismatch_mask, continuity_columns]

station_continuity_breaks = find_station_continuity_breaks(bike_id_to_check)

print(
    f"Liczba przerw w ciągłości stacji dla BikeId={bike_id_to_check}: "
    f"{len(station_continuity_breaks)}"
)

station_continuity_breaks

Liczba przerw w ciągłości stacji dla BikeId=58726.0: 136


,previous_trip_id,previous_end_date,previous_end_station_id,previous_end_station_name,previous_end_lat,previous_end_lon,trip_id,start_date,start_station_id,start_station_name,start_lat,start_lon,end_date,end_station_id,end_station_name,time_since_previous_trip
7955528,136477566.0,2024-01-03 15:28:00,1159.0,"Berry Street, Clerkenwell",51.522853,-0.099994,136490018,2024-01-04 09:56:00,1082,"Borough Road, Elephant & Castle",51.498898,-0.100441,2024-01-04 10:03:00,300231.0,"Lambeth Palace Road, Waterloo",0 days 18:28:00
7885420,136505169.0,2024-01-05 14:55:00,982.0,"Holborn Circus, Holborn",51.517950,-0.108657,136561352,2024-01-09 08:54:00,200057,"Belford House, Haggerston",51.536654,-0.070230,2024-01-09 09:00:00,1047.0,"Falkirk Street, Hoxton",3 days 17:59:00
7808873,136590772.0,2024-01-10 17:26:00,991.0,"Crosswall, Tower",51.511595,-0.077121,136639195,2024-01-13 03:14:00,1060,"Torrens Street, Angel",51.532200,-0.105481,2024-01-13 03:33:00,300040.0,"Ada Street, Hackney Central",2 days 09:48:00
5166800,136714255.0,2024-01-17 08:49:00,200178.0,"Buckingham Gate, Westminster",51.498866,-0.137425,136721856,2024-01-17 14:57:00,3478,"Guildhouse Street, Victoria",51.492346,-0.141334,2024-01-17 15:02:00,1140.0,"Grosvenor Road, Pimlico",0 days 06:08:00
5131145,136755320.0,2024-01-19 07:44:00,22179.0,"Exhibition Road, Knightsbridge",51.499917,-0.174554,136758253,2024-01-19 09:06:00,1182,"Notting Hill Gate Station, Notting Hill",51.509353,-0.196422,2024-01-19 09:21:00,3464.0,"Green Street, Mayfair",0 days 01:22:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11534011,154112961.0,2025-11-27 14:45:00,1011.0,"Argyle Street, Kings Cross",51.529416,-0.123944,154139143,2025-11-28 13:47:00,1009,"Taviton Street, Bloomsbury",51.525051,-0.131161,2025-11-28 13:55:00,300240.0,"Lincoln's Inn Fields, Holborn",0 days 23:02:00
9828405,154405358.0,2025-12-10 12:36:00,984.0,"Finsbury Circus, Liverpool Street",51.517075,-0.086686,154421816,2025-12-10 22:54:00,3427,"Fanshaw Street, Hoxton",51.529537,-0.083353,2025-12-10 22:57:00,200158.0,"Whiston Road, Haggerston",0 days 10:18:00
9795451,154450504.0,2025-12-11 22:47:00,2637.0,"Bermondsey Street, Bermondsey",51.497856,-0.081608,154455832,2025-12-12 08:34:00,1216,"Swan Street, The Borough",51.500296,-0.092763,2025-12-12 08:46:00,200056.0,"Vauxhall Walk, Vauxhall",0 days 09:47:00
10667462,154563529.0,2025-12-17 07:25:00,200128.0,"Queen Street 2, Bank",51.511246,-0.093051,154577517,2025-12-17 16:56:00,999,"Queen Street 1, Bank",51.511553,-0.092940,2025-12-17 17:06:00,1100.0,"Strand, Strand",0 days 09:31:00


In [12]:
# ---------- Implied relocation speed for station continuity breaks ----------

def haversine_distance_km(start_lat, start_lon, end_lat, end_lon):
    start_lat_rad = np.radians(pd.to_numeric(start_lat, errors="coerce"))
    start_lon_rad = np.radians(pd.to_numeric(start_lon, errors="coerce"))
    end_lat_rad = np.radians(pd.to_numeric(end_lat, errors="coerce"))
    end_lon_rad = np.radians(pd.to_numeric(end_lon, errors="coerce"))

    delta_lat = end_lat_rad - start_lat_rad
    delta_lon = end_lon_rad - start_lon_rad

    haversine_a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(start_lat_rad) * np.cos(end_lat_rad) * np.sin(delta_lon / 2) ** 2
    )
    haversine_a = np.clip(haversine_a, 0, 1)

    return 2 * earth_radius_km * np.arcsin(np.sqrt(haversine_a))

break_speed_analysis = station_continuity_breaks.copy()
break_speed_analysis["relocation_distance_km"] = haversine_distance_km(
    break_speed_analysis["previous_end_lat"],
    break_speed_analysis["previous_end_lon"],
    break_speed_analysis["start_lat"],
    break_speed_analysis["start_lon"],
)
break_speed_analysis["relocation_time_hours"] = (
    break_speed_analysis["time_since_previous_trip"] / pd.Timedelta(hours=1)
)
break_speed_analysis["relocation_speed_km_per_h"] = np.where(
    break_speed_analysis["relocation_time_hours"] > 0,
    break_speed_analysis["relocation_distance_km"]
    / break_speed_analysis["relocation_time_hours"],
    np.nan
)

current_trip_speed = ride_distance_data.loc[
    ride_distance_data["trip_id"].isin(break_speed_analysis["trip_id"]),
    ["trip_id", "total_duration_minutes", "straight_line_distance_km"]
].copy()
current_trip_speed["current_trip_speed_km_per_h"] = np.where(
    current_trip_speed["total_duration_minutes"] > 0,
    current_trip_speed["straight_line_distance_km"]
    / (current_trip_speed["total_duration_minutes"] / 60),
    np.nan
)

break_speed_analysis = break_speed_analysis.merge(
    current_trip_speed[
        [
            "trip_id",
            "straight_line_distance_km",
            "total_duration_minutes",
            "current_trip_speed_km_per_h",
        ]
    ],
    on="trip_id",
    how="left",
)

In [13]:
speed_columns = [
    "relocation_distance_km",
    "relocation_time_hours",
    "relocation_speed_km_per_h",
    "current_trip_speed_km_per_h",
]

display(
    break_speed_analysis[speed_columns].describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

display(
    break_speed_analysis[
        [
            "previous_trip_id",
            "previous_end_date",
            "previous_end_station_name",
            "trip_id",
            "start_date",
            "start_station_name",
            "time_since_previous_trip",
            "relocation_distance_km",
            "relocation_speed_km_per_h",
            "current_trip_speed_km_per_h",
        ]
    ]
    .sort_values("relocation_speed_km_per_h", ascending=False)
    .head(20)
)

,relocation_distance_km,relocation_time_hours,relocation_speed_km_per_h,current_trip_speed_km_per_h
count,136.000000,136.000000,136.000000,136.000000
mean,2.192015,18.982353,0.366855,9.904301
std,1.475246,21.750958,0.596619,3.524003
min,0.035013,0.700000,0.001216,0.000000
50%,2.050774,13.483333,0.147295,10.271329
75%,3.114682,24.670833,0.346719,12.329047
90%,4.217643,41.566667,0.978320,13.823286
95%,4.778643,58.908333,1.758800,15.323139
99%,6.014908,87.760833,2.961155,16.661755
max,7.784332,172.183333,3.333150,17.646738


,previous_trip_id,previous_end_date,previous_end_station_name,trip_id,start_date,start_station_name,time_since_previous_trip,relocation_distance_km,relocation_speed_km_per_h,current_trip_speed_km_per_h
77,145561847.0,2025-01-09 10:12:00,"Graham Street, Angel",145562982,2025-01-09 10:59:00,"Regent's Row , Haggerston",0 days 00:47:00,2.610968,3.333150,10.257649
123,152854907.0,2025-10-09 12:08:00,"Soho Square , Soho",152856792,2025-10-09 13:09:00,"St. John's Wood Road, St. John's Wood",0 days 01:01:00,3.202696,3.150193,11.789905
89,147199134.0,2025-03-24 10:59:00,"Kingsway Southbound, Strand",147201250,2025-03-24 12:40:00,"Pritchard's Road, Bethnal Green",0 days 01:41:00,4.393643,2.610085,7.107212
16,137674159.0,2024-03-05 08:31:00,"The Guildhall, Guildhall",137677692,2024-03-05 09:16:00,"Waterloo Station 1, Waterloo",0 days 00:45:00,1.939979,2.586638,11.147486
108,149762413.0,2025-06-25 07:51:00,"Curzon Street, Mayfair",149769030,2025-06-25 09:34:00,"Princes Square, Bayswater",0 days 01:43:00,3.225140,1.878722,10.497095
122,152811189.0,2025-10-08 08:20:00,"Houndsditch, Aldgate",152817446,2025-10-08 09:36:00,"Rectory Square, Stepney",0 days 01:16:00,2.379627,1.878653,9.397729
36,139297859.0,2024-05-13 10:11:00,"Clifford Street, Mayfair",139299919,2024-05-13 11:24:00,"Bayswater Road, Hyde Park",0 days 01:13:00,2.223524,1.827554,7.549818
82,146142572.0,2025-02-06 08:28:00,"St. James's Square, St. James's",146147077,2025-02-06 09:56:00,"Boston Place, Marylebone",0 days 01:28:00,2.545961,1.735882,6.516623
130,154102459.0,2025-11-27 08:38:00,"The Guildhall, Guildhall",154107574,2025-11-27 09:55:00,"Waterloo Station 2, Waterloo",0 days 01:17:00,1.925799,1.500622,9.070487
60,142429574.0,2024-08-29 06:54:00,"St. Bride Street, Holborn",142431347,2024-08-29 07:50:00,"Waterloo Station 2, Waterloo",0 days 00:56:00,1.342413,1.438300,8.927131


In [14]:
high_relocation_speed_threshold_km_h = 25

max_relocation_speed = break_speed_analysis["relocation_speed_km_per_h"].max()
max_current_trip_speed = break_speed_analysis["current_trip_speed_km_per_h"].max()

high_relocation_speed_breaks = (
    break_speed_analysis.loc[
        break_speed_analysis["relocation_speed_km_per_h"]
        >= high_relocation_speed_threshold_km_h,
        [
            "previous_trip_id",
            "previous_end_date",
            "previous_end_station_id",
            "previous_end_station_name",
            "trip_id",
            "start_date",
            "start_station_id",
            "start_station_name",
            "time_since_previous_trip",
            "relocation_distance_km",
            "relocation_time_hours",
            "relocation_speed_km_per_h",
            "current_trip_speed_km_per_h",
        ],
    ]
    .sort_values("relocation_speed_km_per_h", ascending=False)
)

print(
    "Najwyższa wymagana prędkość relokacji: "
    f"{max_relocation_speed:.2f} km/h"
)
print(
    "Najwyższa prędkość kolejnej zarejestrowanej podróży: "
    f"{max_current_trip_speed:.2f} km/h"
)
print(
    "Liczba przerw z wymaganą prędkością relokacji >= "
    f"{high_relocation_speed_threshold_km_h} km/h: "
    f"{len(high_relocation_speed_breaks)}"
)

high_relocation_speed_breaks

Najwyższa wymagana prędkość relokacji: 3.33 km/h
Najwyższa prędkość kolejnej zarejestrowanej podróży: 17.65 km/h
Liczba przerw z wymaganą prędkością relokacji >= 25 km/h: 0


,previous_trip_id,previous_end_date,previous_end_station_id,previous_end_station_name,trip_id,start_date,start_station_id,start_station_name,time_since_previous_trip,relocation_distance_km,relocation_time_hours,relocation_speed_km_per_h,current_trip_speed_km_per_h


In [15]:
# ---------- Station continuity breaks for many bikes ----------

multi_bike_min_trips = 100
multi_bike_max_bikes = 1000  # ustaw None, zeby sprawdzic wszystkie rowery z min. liczba przejazdow
multi_bike_speed_threshold_km_h = high_relocation_speed_threshold_km_h

def find_station_continuity_breaks_for_many_bikes(
    rides=rides_data,
    min_trips_per_bike=multi_bike_min_trips,
    max_bikes_to_check=multi_bike_max_bikes,
    bike_ids=None,
):
    if bike_ids is None:
        selected_bike_counts = rides["bike_id"].value_counts()
        selected_bike_counts = selected_bike_counts[
            selected_bike_counts >= min_trips_per_bike
        ]
        if max_bikes_to_check is not None:
            selected_bike_counts = selected_bike_counts.head(max_bikes_to_check)
        bike_ids_to_check = selected_bike_counts.index
    else:
        bike_ids_to_check = pd.Index(bike_ids)
        selected_bike_counts = rides.loc[
            rides["bike_id"].isin(bike_ids_to_check),
            "bike_id"
        ].value_counts()

    analysis_columns = [
        "trip_id",
        "start_date",
        "start_station_id",
        "start_station_name",
        "end_date",
        "end_station_id",
        "end_station_name",
        "bike_id",
        "start_lat",
        "start_lon",
        "end_lat",
        "end_lon",
    ]

    bike_trips = rides.loc[
        rides["bike_id"].isin(bike_ids_to_check),
        analysis_columns,
    ].copy()

    for coordinate_column in ["start_lat", "start_lon", "end_lat", "end_lon"]:
        bike_trips[coordinate_column] = pd.to_numeric(
            bike_trips[coordinate_column],
            errors="coerce"
        )

    bike_trips = bike_trips.sort_values(
        ["bike_id", "start_date", "end_date", "trip_id"]
    )
    grouped_bike_trips = bike_trips.groupby("bike_id", sort=False)

    bike_trips["previous_trip_id"] = grouped_bike_trips["trip_id"].shift()
    bike_trips["previous_end_date"] = grouped_bike_trips["end_date"].shift()
    bike_trips["previous_end_station_id"] = grouped_bike_trips["end_station_id"].shift()
    bike_trips["previous_end_station_name"] = grouped_bike_trips["end_station_name"].shift()
    bike_trips["previous_end_lat"] = grouped_bike_trips["end_lat"].shift()
    bike_trips["previous_end_lon"] = grouped_bike_trips["end_lon"].shift()
    bike_trips["time_since_previous_trip"] = (
        bike_trips["start_date"] - bike_trips["previous_end_date"]
    )
    bike_trips["bike_trip_count"] = bike_trips["bike_id"].map(selected_bike_counts)

    current_start_station_id = pd.to_numeric(
        bike_trips["start_station_id"],
        errors="coerce"
    )
    previous_end_station_id = pd.to_numeric(
        bike_trips["previous_end_station_id"],
        errors="coerce"
    )

    mismatch_mask = (
        bike_trips["previous_trip_id"].notna()
        & current_start_station_id.notna()
        & previous_end_station_id.notna()
        & (current_start_station_id != previous_end_station_id)
    )

    continuity_columns = [
        "bike_id",
        "bike_trip_count",
        "previous_trip_id",
        "previous_end_date",
        "previous_end_station_id",
        "previous_end_station_name",
        "previous_end_lat",
        "previous_end_lon",
        "trip_id",
        "start_date",
        "start_station_id",
        "start_station_name",
        "start_lat",
        "start_lon",
        "end_date",
        "end_station_id",
        "end_station_name",
        "time_since_previous_trip",
    ]

    return (
        bike_trips.loc[mismatch_mask, continuity_columns].reset_index(drop=True),
        selected_bike_counts,
    )

multi_bike_station_continuity_breaks, multi_bike_selected_counts = (
    find_station_continuity_breaks_for_many_bikes()
)

print(f"Analizowane rowery: {len(multi_bike_selected_counts)}")
print(
    "Rowery z co najmniej jedna przerwa ciaglosci stacji: "
    f"{multi_bike_station_continuity_breaks['bike_id'].nunique()}"
)
print(f"Liczba przerw ciaglosci stacji: {len(multi_bike_station_continuity_breaks)}")

multi_bike_station_continuity_breaks.head()

Analizowane rowery: 1000
Rowery z co najmniej jedna przerwa ciaglosci stacji: 1000
Liczba przerw ciaglosci stacji: 108009


,bike_id,bike_trip_count,previous_trip_id,previous_end_date,previous_end_station_id,previous_end_station_name,previous_end_lat,previous_end_lon,trip_id,start_date,start_station_id,start_station_name,start_lat,start_lon,end_date,end_station_id,end_station_name,time_since_previous_trip
0,50017.0,1827,136454676.0,2024-01-01 12:47:00,200043.0,"Bishop's Avenue, Fulham",51.473037,-0.214750,136483021,2024-01-03 20:20:00,200098,"Aintree Street, Fulham",51.481021,-0.209973,2024-01-03 20:33:00,300045.0,"Albert Bridge Road, Battersea Park",2 days 07:33:00
1,50017.0,1827,136485053.0,2024-01-04 07:25:00,1161.0,"Brushfield Street, Liverpool Street",51.518908,-0.079249,136519094,2024-01-06 13:44:00,200003,"Cheshire Street, Bethnal Green",51.523880,-0.065076,2024-01-06 13:58:00,1131.0,"Graham Street, Angel",2 days 06:19:00
2,50017.0,1827,136603611.0,2024-01-11 09:19:00,999.0,"Queen Street 1, Bank",51.511553,-0.092940,136612045,2024-01-11 17:07:00,200128,"Queen Street 2, Bank",51.511246,-0.093051,2024-01-11 17:18:00,1072.0,"Waterloo Station 3, Waterloo",0 days 07:48:00
3,50017.0,1827,136612045.0,2024-01-11 17:18:00,1072.0,"Waterloo Station 3, Waterloo",51.503792,-0.112824,136632005,2024-01-12 17:05:00,200094,"St. Bride Street, Holborn",51.514665,-0.104584,2024-01-12 17:11:00,22163.0,"Southwark Station 2, Southwark",0 days 23:47:00
4,50017.0,1827,136693840.0,2024-01-16 09:30:00,999.0,"Queen Street 1, Bank",51.511553,-0.092940,136717628,2024-01-17 10:03:00,1005,"Eversholt Street , Camden Town",51.533005,-0.136793,2024-01-17 10:09:00,1009.0,"Taviton Street, Bloomsbury",1 days 00:33:00


In [16]:
# ---------- Implied relocation speed for many bikes ----------

multi_bike_break_speed_analysis = multi_bike_station_continuity_breaks.copy()
multi_bike_break_speed_analysis["relocation_distance_km"] = haversine_distance_km(
    multi_bike_break_speed_analysis["previous_end_lat"],
    multi_bike_break_speed_analysis["previous_end_lon"],
    multi_bike_break_speed_analysis["start_lat"],
    multi_bike_break_speed_analysis["start_lon"],
)
multi_bike_break_speed_analysis["relocation_time_hours"] = (
    multi_bike_break_speed_analysis["time_since_previous_trip"] / pd.Timedelta(hours=1)
)
multi_bike_break_speed_analysis["relocation_speed_km_per_h"] = np.where(
    multi_bike_break_speed_analysis["relocation_time_hours"] > 0,
    multi_bike_break_speed_analysis["relocation_distance_km"]
    / multi_bike_break_speed_analysis["relocation_time_hours"],
    np.nan,
)

multi_bike_current_trip_speed = ride_distance_data.loc[
    ride_distance_data["trip_id"].isin(multi_bike_break_speed_analysis["trip_id"]),
    ["trip_id", "total_duration_minutes", "straight_line_distance_km"]
].copy()
multi_bike_current_trip_speed["current_trip_speed_km_per_h"] = np.where(
    multi_bike_current_trip_speed["total_duration_minutes"] > 0,
    multi_bike_current_trip_speed["straight_line_distance_km"]
    / (multi_bike_current_trip_speed["total_duration_minutes"] / 60),
    np.nan,
)

multi_bike_break_speed_analysis = multi_bike_break_speed_analysis.merge(
    multi_bike_current_trip_speed[
        [
            "trip_id",
            "straight_line_distance_km",
            "total_duration_minutes",
            "current_trip_speed_km_per_h",
        ]
    ],
    on="trip_id",
    how="left",
)

display(
    multi_bike_break_speed_analysis[
        [
            "relocation_distance_km",
            "relocation_time_hours",
            "relocation_speed_km_per_h",
            "current_trip_speed_km_per_h",
        ]
    ].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
)

multi_bike_break_speed_analysis[
    [
        "bike_id",
        "previous_end_date",
        "previous_end_station_name",
        "start_date",
        "start_station_name",
        "time_since_previous_trip",
        "relocation_distance_km",
        "relocation_speed_km_per_h",
        "current_trip_speed_km_per_h",
    ]
].sort_values("relocation_speed_km_per_h", ascending=False).head(20)

,relocation_distance_km,relocation_time_hours,relocation_speed_km_per_h,current_trip_speed_km_per_h
count,108009.000000,108009.000000,108008.000000,108009.000000
mean,2.483632,36.747309,0.345289,9.895708
std,1.696538,88.970893,0.571625,3.692875
min,0.013742,-0.100000,0.000191,0.000000
50%,2.141699,15.033333,0.138147,10.370470
75%,3.241778,28.783333,0.387623,12.264584
90%,4.671633,72.983333,0.915665,14.025904
95%,5.808741,142.050000,1.388304,15.082691
99%,8.108321,427.632000,2.721672,17.126455
max,15.743855,3356.983333,35.860961,50.026967


,bike_id,previous_end_date,previous_end_station_name,start_date,start_station_name,time_since_previous_trip,relocation_distance_km,relocation_speed_km_per_h,current_trip_speed_km_per_h
25559,56816.0,2025-10-26 01:13:00,"Thurtle Road, Haggerston",2025-10-26 01:23:00,"Curzon Street, Mayfair",0 days 00:10:00,5.976827,35.860961,13.754765
42028,58283.0,2025-09-29 10:05:00,"Putney Rail Station, Putney",2025-09-29 10:41:00,"Vauxhall Cross, Vauxhall",0 days 00:36:00,6.964485,11.607476,18.755507
56501,58898.0,2024-04-21 15:09:00,"Smith Square, Westminster",2024-04-21 15:40:00,"Alma Road, Wandsworth",0 days 00:31:00,5.778425,11.184048,7.698085
90102,59715.0,2024-10-07 08:53:00,"Crosswall, Tower",2024-10-07 09:19:00,"Green Park Station, Mayfair",0 days 00:26:00,4.582061,10.573986,0.000000
25981,56836.0,2025-04-19 17:02:00,"Wright's Lane, Kensington",2025-04-19 17:16:00,"Humbolt Road, Fulham",0 days 00:14:00,2.311353,9.905797,11.213168
25031,56781.0,2024-07-25 09:41:00,"Palissy Street, Shoreditch",2024-07-25 09:48:00,"Pott Street, Bethnal Green",0 days 00:07:00,1.107547,9.493260,6.452084
107,50017.0,2025-12-05 13:20:00,"South Park, Sands End",2025-12-05 13:32:00,"Finlay Street, Fulham",0 days 00:12:00,1.896918,9.484591,0.000000
38398,58090.0,2024-10-07 17:58:00,"Finsbury Circus, Liverpool Street",2024-10-07 18:49:00,"Bramham Gardens, Earl's Court",0 days 00:51:00,7.776932,9.149332,0.000000
26675,56925.0,2025-08-14 08:54:00,"Brushfield Street, Liverpool Street",2025-08-14 09:20:00,"Mostyn Grove, Bow",0 days 00:26:00,3.937384,9.086270,10.389829
66280,59222.0,2024-12-17 02:20:00,"Smugglers Way, Wandsworth",2024-12-17 03:13:00,"Poured Lines, Bankside",0 days 00:53:00,7.959111,9.010315,3.962961


In [17]:
# ---------- Multi-bike anomaly ranking ----------

multi_bike_possible_transports = (
    multi_bike_break_speed_analysis.loc[
        multi_bike_break_speed_analysis["relocation_speed_km_per_h"]
        >= multi_bike_speed_threshold_km_h,
        [
            "bike_id",
            "bike_trip_count",
            "previous_trip_id",
            "previous_end_date",
            "previous_end_station_id",
            "previous_end_station_name",
            "trip_id",
            "start_date",
            "start_station_id",
            "start_station_name",
            "time_since_previous_trip",
            "relocation_distance_km",
            "relocation_time_hours",
            "relocation_speed_km_per_h",
            "current_trip_speed_km_per_h",
        ],
    ]
    .sort_values("relocation_speed_km_per_h", ascending=False)
)

multi_bike_anomaly_summary = (
    multi_bike_break_speed_analysis
    .groupby("bike_id")
    .agg(
        bike_trip_count=("bike_trip_count", "max"),
        continuity_break_count=("trip_id", "count"),
        high_speed_break_count=(
            "relocation_speed_km_per_h",
            lambda speeds: (speeds >= multi_bike_speed_threshold_km_h).sum()
        ),
        max_relocation_speed_km_h=("relocation_speed_km_per_h", "max"),
        p95_relocation_speed_km_h=(
            "relocation_speed_km_per_h",
            lambda speeds: speeds.quantile(0.95)
        ),
        max_relocation_distance_km=("relocation_distance_km", "max"),
        median_relocation_time_h=("relocation_time_hours", "median"),
    )
    .sort_values(
        ["high_speed_break_count", "max_relocation_speed_km_h"],
        ascending=False
    )
)

print(
    "Liczba mozliwych transportow rowerow przy progu >= "
    f"{multi_bike_speed_threshold_km_h} km/h: {len(multi_bike_possible_transports)}"
)
print(
    "Rowery z mozliwym transportem: "
    f"{multi_bike_possible_transports['bike_id'].nunique()}"
)

display(multi_bike_anomaly_summary.head(20))
multi_bike_possible_transports.head(50)

Liczba mozliwych transportow rowerow przy progu >= 25 km/h: 1
Rowery z mozliwym transportem: 1


,bike_trip_count,continuity_break_count,high_speed_break_count,max_relocation_speed_km_h,p95_relocation_speed_km_h,max_relocation_distance_km,median_relocation_time_h
bike_id,,,,,,,
56816.0,1896,120,1,35.860961,1.113139,8.338662,17.416667
58283.0,1883,88,0,11.607476,1.289296,7.690043,15.358333
58898.0,1968,115,0,11.184048,1.417103,10.862091,15.016667
59715.0,2309,136,0,10.573986,1.571633,6.299842,12.075000
56836.0,2042,102,0,9.905797,1.484844,8.410459,20.433333
56781.0,1939,121,0,9.493260,2.530257,8.224215,14.933333
50017.0,1827,111,0,9.484591,1.572939,10.388074,18.983333
58090.0,1897,100,0,9.149332,1.357404,9.941783,15.708333
56925.0,2200,124,0,9.086270,1.833930,7.582463,11.166667


,bike_id,bike_trip_count,previous_trip_id,previous_end_date,previous_end_station_id,previous_end_station_name,trip_id,start_date,start_station_id,start_station_name,time_since_previous_trip,relocation_distance_km,relocation_time_hours,relocation_speed_km_per_h,current_trip_speed_km_per_h
25559,56816.0,1896,153307415.0,2025-10-26 01:13:00,200093.0,"Thurtle Road, Haggerston",153307241,2025-10-26 01:23:00,3437,"Curzon Street, Mayfair",0 days 00:10:00,5.976827,0.166667,35.860961,13.754765


In [18]:
# ---------- Readable summary of suspicious bike relocations ----------

def format_identifier(value):
    if pd.isna(value):
        return "brak"
    return str(int(value))

if multi_bike_possible_transports.empty:
    print(
        "Brak podejrzanych relokacji przy progu >= "
        f"{multi_bike_speed_threshold_km_h} km/h."
    )
else:
    print(
        "Podejrzane relokacje rowerow przy progu >= "
        f"{multi_bike_speed_threshold_km_h} km/h: "
        f"{len(multi_bike_possible_transports)}"
    )

    for anomaly_number, (_, suspicious_row) in enumerate(
        multi_bike_possible_transports.iterrows(),
        start=1
    ):
        print("\n" + "=" * 80)
        print(f"Podejrzana relokacja #{anomaly_number}")
        print(f"BikeId: {format_identifier(suspicious_row['bike_id'])}")
        print(
            "Tripy: "
            f"{format_identifier(suspicious_row['previous_trip_id'])} -> "
            f"{format_identifier(suspicious_row['trip_id'])}"
        )
        print(
            "Stacje: "
            f"{suspicious_row['previous_end_station_name']} -> "
            f"{suspicious_row['start_station_name']}"
        )
        print(
            "Czas miedzy przejazdami: "
            f"{suspicious_row['time_since_previous_trip']}"
        )
        print(
            "Wymagana predkosc relokacji: "
            f"{suspicious_row['relocation_speed_km_per_h']:.2f} km/h"
        )
        print(
            "Dystans relokacji w linii prostej: "
            f"{suspicious_row['relocation_distance_km']:.2f} km"
        )

Podejrzane relokacje rowerow przy progu >= 25 km/h: 1

Podejrzana relokacja #1
BikeId: 56816
Tripy: 153307415 -> 153307241
Stacje: Thurtle Road, Haggerston -> Curzon Street, Mayfair
Czas miedzy przejazdami: 0 days 00:10:00
Wymagana predkosc relokacji: 35.86 km/h
Dystans relokacji w linii prostej: 5.98 km


In [19]:
# ---------- Dataset-wide station continuity break count ----------

dataset_sequence_columns = [
    "bike_id",
    "trip_id",
    "start_date",
    "end_date",
    "start_station_id",
    "end_station_id",
]

dataset_trip_sequence = (
    rides_data[dataset_sequence_columns]
    .dropna(subset=["bike_id"])
    .sort_values(["bike_id", "start_date", "end_date", "trip_id"])
    .copy()
)

dataset_grouped_trips = dataset_trip_sequence.groupby("bike_id", sort=False)
dataset_trip_sequence["previous_trip_id"] = dataset_grouped_trips["trip_id"].shift()
dataset_trip_sequence["previous_end_station_id"] = (
    dataset_grouped_trips["end_station_id"].shift()
)

dataset_current_start_station_id = pd.to_numeric(
    dataset_trip_sequence["start_station_id"],
    errors="coerce"
)
dataset_previous_end_station_id = pd.to_numeric(
    dataset_trip_sequence["previous_end_station_id"],
    errors="coerce"
)

dataset_has_previous_trip_mask = dataset_trip_sequence["previous_trip_id"].notna()
dataset_comparable_station_mask = (
    dataset_has_previous_trip_mask
    & dataset_current_start_station_id.notna()
    & dataset_previous_end_station_id.notna()
)
dataset_station_break_mask = (
    dataset_comparable_station_mask
    & (dataset_current_start_station_id != dataset_previous_end_station_id)
)

total_trip_count = len(rides_data)
trips_with_previous_trip_count = int(dataset_has_previous_trip_mask.sum())
comparable_trip_count = int(dataset_comparable_station_mask.sum())
dataset_station_break_count = int(dataset_station_break_mask.sum())

break_share_all_trips = 100 * dataset_station_break_count / total_trip_count
break_share_trips_with_previous = (
    100 * dataset_station_break_count / trips_with_previous_trip_count
)
break_share_comparable_trips = 100 * dataset_station_break_count / comparable_trip_count

print("Przerwy ciaglosci stacji w calym zbiorze, liczone w ramach tego samego BikeId")
print(f"Wszystkie przejazdy: {total_trip_count:,}")
print(f"Przejazdy z poprzednim przejazdem tego samego roweru: {trips_with_previous_trip_count:,}")
print(f"Porownywalne przejazdy z niepustymi stacjami: {comparable_trip_count:,}")
print(f"Liczba przerw ciaglosci stacji: {dataset_station_break_count:,}")
print(f"Procent wszystkich przejazdow: {break_share_all_trips:.2f}%")
print(
    "Procent przejazdow z poprzednim przejazdem tego samego roweru: "
    f"{break_share_trips_with_previous:.2f}%"
)
print(
    "Procent porownywalnych przejazdow: "
    f"{break_share_comparable_trips:.2f}%"
)

Przerwy ciaglosci stacji w calym zbiorze, liczone w ramach tego samego BikeId
Wszystkie przejazdy: 17,426,293
Przejazdy z poprzednim przejazdem tego samego roweru: 17,410,554
Porownywalne przejazdy z niepustymi stacjami: 17,410,554
Liczba przerw ciaglosci stacji: 1,044,959
Procent wszystkich przejazdow: 6.00%
Procent przejazdow z poprzednim przejazdem tego samego roweru: 6.00%
Procent porownywalnych przejazdow: 6.00%
